<a href="https://colab.research.google.com/github/tanjun8802/Mase_EDGE/blob/jtan%2Fdev/EDGE_NAS_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mase_EDGE - Optuna Study and Executorch export Pipeline

Mase EDGE performs Optuna study for a specific Android phone that connects to the host PC. Alongside optimisation, the Optuna study will gather the phone information obtained through adb.exe (Android Developer mode) and define them as metrics to be studied. For example, based on the available hardware resources on the phone, alter the delegation ratio between the hardware for optimal performance development.

In [ ]:
# # For colab use

# # !git clone --branch jtan/dev https://github.com/tanjun8802/Mase_EDGE.git 
# # %cd /content
# # !rm -rf Mase_EDGE/mase
# # %cd Mase_EDGE
# !git submodule update --init --recursive
# %cd mase/
# !pip install -e .

In [ ]:
# # Get the dataset (Tiny ImageNet)

# %cd /content/Mase_EDGE
# !wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
# !unzip tiny-imagenet-200.zip

In [ ]:
import optuna
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.utils.prune as prune
import torch.export as exir
import torchvision.transforms as T
import torch.optim as optim
import subprocess
import json
import time
from tqdm.auto import tqdm
import re
import os, sys
from pathlib import Path
from typing import Any, Dict, Optional
from torchvision import datasets
from mase.src.chop import MaseGraph
import mase.src.chop.passes as passes

# Jupyter has no __file__; assume the kernel cwd is the repo root (ADLSProject).
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from EDGE_device.device_specifications import (
    get_adb_output,
    get_adb_sh_c,
    get_hardware_for_phone,
    load_phone_specs,
)


In [ ]:
# Android applicationId — must match app/build.gradle.kts defaultConfig.applicationId
IMAGE_CLASSIFICATION_APP_ID = "com.image_classification_app"

# Host path to a folder with train/, val/, test/ (pushed on trial 0 only). If None, no dataset
# is sent and the app writes error metrics (acc=0, lat=999) — easy to mistake for "bad polling".
_default_ds = PROJECT_ROOT / "mase" / "tiny-imagenet-200"
EDGE_DEVICE_DATASET_PATH: Optional[str] = (
    str(_default_ds) if _default_ds.is_dir() else None
)
if EDGE_DEVICE_DATASET_PATH:
    print(f"EDGE_DEVICE_DATASET_PATH → {EDGE_DEVICE_DATASET_PATH}")
else:
    print(
        "WARN: No dataset path (cwd/mase/tiny-imagenet-200 missing). "
        "Trial 0 will not push data; expect acc=0 / lat=999 from the app."
    )
# Which split the app evaluates: "train" | "val" | "test"
EDGE_EVAL_SPLIT: str = "val"

# Full Tiny ImageNet val can take tens of minutes on device; override via env.
EDGE_METRICS_POLL_TIMEOUT_FLAG: int = 3600
EDGE_BENCHMARK_MAX_IMAGES_FLAG: Optional[int] = 10  # None = no cap on device eval images
BASE_STATE_DICT = None  # CPU weight snapshot after one-time train; fresh nn.Module each trial

# Base ResNet18 weights: python notebooks/resnet18_qat_fp32.pt (copy or export there)
RESNET18_PT_PATH = PROJECT_ROOT / "python notebooks" / "resnet18_qat_fp32.pt"
# PTQ/QAT training if used
EPOCHS = 1
LR = 1e-4
BATCH_SIZE = 64

### 1. Load Metrics from Android Phone

If you are trying to optimise and deploy the model on an Android phone, please plug in the phone to the laptop and make sure the developer mode is on. In settings make sure debugging mode is turned on. Run the following code cell to extract the hardware information. A JSON file will be created to hold the information and it will be used in the configuration setup.

In [ ]:
hw = get_hardware_for_phone() # Make sure the phone is connected and ADB is set up
print("Optimizing for:", hw["model"])
print(hw["gpu"], hw["cpu_cores"], hw["ram_gb"])

In [ ]:
# NPU/NNAPI hardware check (non-reference devices indicate accelerators like NPU)
# Reload so notebook picks up latest ``get_adb_sh_c`` (e.g. ``timeout_sec``) without kernel restart.
import importlib
import EDGE_device.device_specifications as _adb_spec_mod
importlib.reload(_adb_spec_mod)
get_adb_output = _adb_spec_mod.get_adb_output
get_adb_sh_c = _adb_spec_mod.get_adb_sh_c

# ``get_adb_output`` / ``get_adb_sh_c`` use a timeout (default 20s, env ``ADB_SHELL_TIMEOUT_SEC``).
# Avoid ``service list`` here — it often blocks for a long time on real devices.
checks_simple = {
    "ro.hardware": "shell getprop ro.hardware",
}
checks_shell = {
    # ``grep -i npu`` false-positives on ``c2inputsurface`` (substring ``input`` → n-p-u)
    "npu_prop": "getprop | grep -i npu | grep -vi c2inputsurface || true",
    "cpuinfo": "head -20 /proc/cpuinfo",
    "nn_hal": "ls /vendor/lib/hw/ /vendor/lib64/hw/ 2>/dev/null | grep neuralnetworks || echo none",
    "init_neural_svcs": "getprop | grep -i init.svc.neural || true",
}

results = {k: get_adb_output(v) for k, v in checks_simple.items()}
for k, script in checks_shell.items():
    results[k] = get_adb_sh_c(script, timeout_sec=15.0)

# Heuristic NPU detection
npu_indicators = ["npu", "hexagon", "qcom", "kirin", "mali", "neuralnetworks"]
has_npu = any(any(ind in res.lower() for ind in npu_indicators) for res in results.values())
print(f"NPU/Accel likely: {'YES' if has_npu else 'NO'}")
print("Full results:", results)

In [ ]:
specs = load_phone_specs('EDGE_device/device_specifications.json')          # uses default PHONE_SPECS_FILE
print("Specs dict:", specs)
print("Keys:", list(specs.keys()))

if specs:
    hw = list(specs.values())[0]
    print(hw["model"], hw["gpu"], hw["cpu_cores"], hw["ram_gb"])
else:
    print("No specs found; JSON file missing or empty.")

### 2. Defining the Search Space

Here the code prepares the configurations for the pipeline

In [ ]:
def edge_optuna_config(trial):
    """Hardware-aware config: compression + mixed delegation."""

    # base_opts = ["cpu"]
    # if hw.get("prefers_gpu"):
    #     base_opts.extend(["xnnpack", "vulkan"])  # CPU/GPU
    # if hw.get("has_npu"):
    #     base_opts.extend(["qnn", "nnapi"])       # NPU
    # base_opts = ["xnnpack"]  # unused while primary_backend is pinned below

    # Primary backend (where most ops go)
    # primary_backend = trial.suggest_categorical("primary_backend", base_opts)
    primary_backend = "xnnpack"

    # Whether to use mixed delegation at all
    # use_mixed = trial.suggest_categorical("use_mixed_delegation", [False, True])
    use_mixed = False  # XNNPACK-only study: keep export path simple

    # Compression 
    prune_ratio = trial.suggest_float("prune_ratio", 0.0, 0.7)
    quant_bits = trial.suggest_categorical("quant_bits", [8, 16])

    # Global default quant config
    if primary_backend in ["vulkan", "xnnpack"]:
        global_quant = trial.suggest_categorical("quant_config_global", ["int8", "fp16"])
    else:
        global_quant = "int8"

    # If not mixing, just return simple config
    if not use_mixed:
        return {
            "prune_ratio": prune_ratio,
            "quant_bits": quant_bits,
            "backend": primary_backend,
            "use_mixed_delegation": False,
            "delegation_plan": None,
            "quant_config_global": global_quant,
        }


    # Which units are allowed in the mix
    allowed_units = ["cpu"]
    # if hw.get("prefers_gpu"):
    #     allowed_units.append("gpu")
    # if hw.get("has_npu"):
    #     allowed_units.append("npu")

    # Choose a high-level “pattern” for splitting
    pattern = trial.suggest_categorical(
        "delegation_pattern",
        [
            "encoder_npu_decoder_cpu",     # early layers on NPU, later on CPU
            "early_layers_npu_late_cpu",  # similar but more flexible
            "attention_npu_rest_gpu", # attention on NPU, rest on GPU
            "gpu_primary_npu_offload", # give some to npu
            "cpu_primary_gpu_offload", # give some to gpu
        ],
    )

    # Per-unit ratios (0–1) – you can interpret them as fractions of ops or layers
    # and normalize in your partitioner to achieve load distribution. Optuna will search for the best balance.
    cpu_ratio = trial.suggest_float("cpu_ratio_raw", 0.0, 1.0)
    gpu_ratio = trial.suggest_float("gpu_ratio_raw", 0.0, 1.0) if "gpu" in allowed_units else 0.0
    npu_ratio = trial.suggest_float("npu_ratio_raw", 0.0, 1.0) if "npu" in allowed_units else 0.0

    # Will the partitioner auto round this in subgraph split?

    # Normalize
    total = cpu_ratio + gpu_ratio + npu_ratio
    if total == 0:
        cpu_ratio_norm, gpu_ratio_norm, npu_ratio_norm = 1.0, 0.0, 0.0
    else:
        cpu_ratio_norm = cpu_ratio / total
        gpu_ratio_norm = gpu_ratio / total
        npu_ratio_norm = npu_ratio / total

   
    # If consider NPU “high” more aggressive int8 there
    if hw.get("has_npu") and hw.get("npu_compute_level") == "high":
        npu_quant = trial.suggest_categorical("npu_quant_config", ["int8"])
    elif hw.get("has_npu"):
        npu_quant = trial.suggest_categorical("npu_quant_config", ["int8", "fp16"])
    else:
        npu_quant = None

    gpu_quant = None
    if hw.get("prefers_gpu"):
        gpu_quant = trial.suggest_categorical("gpu_quant_config", ["fp16", "int8"])

    cpu_quant = trial.suggest_categorical("cpu_quant_config", ["int8", "fp16"])

    delegation_plan = {
        "pattern": pattern,
        "ratios": {
            "cpu": cpu_ratio_norm,
            "gpu": gpu_ratio_norm,
            "npu": npu_ratio_norm,
        },
        "unit_quant": {
            "cpu": cpu_quant,
            "gpu": gpu_quant,
            "npu": npu_quant,
        },
    }

    return {
        "prune_ratio": prune_ratio,
        "quant_bits": quant_bits,
        "backend": primary_backend,
        "use_mixed_delegation": True,
        "delegation_plan": delegation_plan,
        "quant_config_global": global_quant,
    }

### 3. Model Optimisation

In [ ]:
# Inherit from Clarence's notebook and mase is used here


def _load_resnet18_from_pt(pt_path: Path, load_device: str) -> nn.Module:
    """Load ResNet18 from .pt: dict+state_dict, raw state_dict, or full nn.Module (torch.save(model))."""
    if not pt_path.is_file():
        raise FileNotFoundError(
            f"Missing {pt_path!s}. Add resnet18_qat_fp32.pt under python notebooks/ (e.g. export from ResNet18.ipynb)."
        )
    ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)
    n_cls = 200
    if isinstance(ckpt, nn.Module):
        sd = ckpt.state_dict()
        if hasattr(ckpt, "fc") and isinstance(ckpt.fc, nn.Linear):
            n_cls = int(ckpt.fc.out_features)
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        inner = ckpt["state_dict"]
        n_cls = int(ckpt.get("num_classes", 200))
        if isinstance(inner, nn.Module):
            sd = inner.state_dict()
            if hasattr(inner, "fc") and isinstance(inner.fc, nn.Linear):
                n_cls = int(inner.fc.out_features)
        else:
            sd = inner
    elif isinstance(ckpt, dict):
        sd = ckpt
        if "fc.bias" in sd:
            n_cls = int(sd["fc.bias"].shape[0])
    else:
        raise TypeError(f"Unsupported checkpoint type {type(ckpt)}; expected dict or nn.Module.")
    if not isinstance(sd, dict):
        raise TypeError(f"Expected state_dict to be dict-like, got {type(sd)}.")
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, n_cls)
    model.load_state_dict(sd, strict=True)
    return model.to(load_device)


def load_model_and_train() -> nn.Module:
    """Train once; every call builds a new ResNet from BASE_STATE_DICT (Mase prune/FX must not reuse module objects)."""
    global BASE_STATE_DICT, LR, EPOCHS, BATCH_SIZE
    if BASE_STATE_DICT is None:
        print(
            f"LOADING BASE WEIGHTS FROM {RESNET18_PT_PATH.name} AND TRAINING FOR {EPOCHS} EPOCH"
        )
        train_root = PROJECT_ROOT / "mase" / "tiny-imagenet-200" / "train"
        
        load_device = ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
        else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        
        model = _load_resnet18_from_pt(RESNET18_PT_PATH, load_device)
        
        # criterion = nn.CrossEntropyLoss()
        # optimizer = optim.Adam(model.parameters(), lr=LR)
        # transform = T.Compose(
        #     [
        #         T.Resize(224),
        #         T.ToTensor(),
        #         T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        #     ]
        # )
        # train_loader = torch.utils.data.DataLoader(
        #     datasets.ImageFolder(root=str(train_root), transform=transform),
        #     batch_size=BATCH_SIZE,
        #     shuffle=True,
        #     num_workers=0,
        # )
        # model.train()
        # for ep in range(EPOCHS):
        #     for imgs, labels in tqdm(
        #         train_loader,
        #         desc=f"train epoch {ep + 1}/{EPOCHS}",
        #     ):
        #         imgs, labels = imgs.to(load_device), labels.to(load_device)
        #         optimizer.zero_grad()
        #         loss = criterion(model(imgs), labels)
        #         loss.backward()
        #         optimizer.step()
        model.eval()
        print("Model is prepared.....")
        BASE_STATE_DICT = {
            k: v.detach().cpu().clone() for k, v in model.state_dict().items()
        }
    load_device = (
        "mps"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
        else ("cuda" if torch.cuda.is_available() else "cpu")
    )
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 200)
    m.load_state_dict(BASE_STATE_DICT, strict=True)
    return m.to(load_device)


def edge_optimise_model(
    config: Dict[str, Any],
    enable_qat: bool = False,
    model: Optional[nn.Module] = None,
) -> nn.Module:
    """Prune → integer quant; optional QAT on Tiny ImageNet train if enable_qat"""
    if model is None:
        model = load_model_and_train()
    global LR, EPOCHS, BATCH_SIZE
    qat_device =  ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
        else ("cuda" if torch.cuda.is_available() else "cpu")
        )
    
    qb = int(config["quant_bits"])
    frac = max(0, qb - 1)
    layer_quant = {
        "name": "integer",
        "data_in_width": qb,
        "data_in_frac_width": frac,
        "weight_width": qb,
        "weight_frac_width": frac,
        "bias_width": 32,
        "bias_frac_width": 0,
    }
    
    example_inputs = torch.randn(1, 3, 224, 224).to(qat_device)
    dummy_inputs = {"x": example_inputs}

    mg = MaseGraph(model)
    mg, _ = passes.init_metadata_analysis_pass(mg)
    mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": dummy_inputs})

    pruning_config = {
        "weight": {
            "sparsity": config["prune_ratio"],
            "method": "l1-norm",
            "scope": "local",
        },
        "activation": {
            "sparsity": config["prune_ratio"],
            "method": "l1-norm",
            "scope": "local",
        },
    }
    mg, _ = passes.prune_transform_pass(mg, pass_args=pruning_config)

    # Integer modules use STE-style quant
    quantization_config = {
        "by": "type",
        "default": {"config": {"name": None}},
        "conv2d": {"config": dict(layer_quant)},
        "linear": {"config": dict(layer_quant)},
    }
    mg, _ = passes.quantize_transform_pass(mg, pass_args=quantization_config)

    gm = torch.fx.GraphModule(mg.model, mg.fx_graph)

    train_root = PROJECT_ROOT / "mase" / "tiny-imagenet-200" / "train"

    if enable_qat:
        gm = gm.to(qat_device)
        if train_root.is_dir():
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(gm.parameters(), lr=LR)
            transform = T.Compose(
                [
                    T.Resize(224),
                    T.ToTensor(),
                    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
                ]
            )
            train_loader = torch.utils.data.DataLoader(
                datasets.ImageFolder(root=str(train_root), transform=transform),
                batch_size=BATCH_SIZE,
                shuffle=True,
                num_workers=0,
            )
            gm.train()
            for ep in range(EPOCHS):
                for imgs, labels in tqdm(
                    train_loader,
                    desc=f"QAT train epoch {ep + 1}/{EPOCHS} ({qb}-bit, after prune)",
                ):
                    imgs, labels = imgs.to(qat_device), labels.to(qat_device)
                    optimizer.zero_grad()
                    loss = criterion(gm(imgs), labels)
                    loss.backward()
                    optimizer.step()
            gm.eval()
        else:
            print(
                f"WARN: {train_root} missing — skipping QAT steps; "
                "quantized model not fine-tuned (accuracy may be poor)."
            )
            gm.eval()
    else:
        print(
            "No QAT was done, performance may be poor check the edge_host_val_sanity_check result"
        )
        gm.eval()

    return gm.cpu()


def edge_host_val_sanity_check(
    model: nn.Module,
    *,
    train_dir: Optional[Path] = None,
    max_batches: int = 10,
    batch_size: int = 64,
) -> Optional[float]:
    """Mean top-1 over a few Tiny ImageNet *train* ImageFolder batches (same class order as Android)."""
    root = train_dir or (PROJECT_ROOT / "mase" / "tiny-imagenet-200" / "train")
    if not root.is_dir():
        return None
    transform = T.Compose(
        [
            T.Resize(224),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )
    loader = torch.utils.data.DataLoader(
        datasets.ImageFolder(root=str(root), transform=transform),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )
    model.eval()
    device = next(model.parameters()).device
    correct = total = 0
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(loader):
            if i >= max_batches:
                break
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            pred = logits.argmax(dim=1)
            correct += int((pred == labels).sum().item())
            total += int(labels.numel())
    return (correct / total) if total else None



### 4. Export .pte file for Edge Device

In [ ]:
from pathlib import Path
from typing import Any, Dict

import torch
import torch.nn as nn

import executorch.exir as exir
from executorch.exir import to_edge

from executorch.exir.backend.backend_api import to_backend
from executorch.exir import ExecutorchProgramManager

# Import concrete partitioners
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
# from executorch.backends.vulkan.partition.vulkan_partitioner import VulkanPartitioner
# from executorch.backends.qnn.partition.qnn_partitioner import QnnPartitioner
# from executorch.backends.nnapi.partition.nnapi_partitioner import NnapiPartitioner
# For CPU-only you can just skip a backend, or define a trivial one if needed.


def edge_export_model(model: nn.Module, trial_id: int, config: Dict[str, Any]) -> str:
    """Export model to ExecuTorch .pte with backend partitioning."""

    model.eval()
    example_input = torch.randn(1, 3, 224, 224)

    exported = torch.export.export(model, (example_input,))
    edge: exir.EdgeProgramManager = to_edge(exported)

    # 3) Choose partitioner(s) based on config["backend"]
    backend = config["backend"]

    # partitioners = []
    # if backend == "xnnpack":
    #     partitioners.append(XnnpackPartitioner())
    # elif backend == "vulkan":
    #     partitioners.append(VulkanPartitioner())
    # elif backend == "qnn":
    #     partitioners.append(QnnPartitioner())
    # elif backend == "nnapi":
    #     partitioners.append(NnapiPartitioner())
    # elif backend == "cpu":
    #     # no delegation, leave as pure edge (no partitioner).
    #     partitioners = []
    # else:
    #     # Fallback to XNNPACK
    #     partitioners.append(XnnpackPartitioner())
    partitioners = [XnnpackPartitioner()]

    # 4) Apply partitioners (if any)
    #    Note: you can chain multiple partitioners if you want mixed delegation.
    edge_prog = edge
    for part in partitioners:
        edge_prog = edge_prog.to_backend(part)

    # fusion would go here on edge_prog


    # Convert to ExecuTorch program and write .pte
    exec_prog: ExecutorchProgramManager = edge_prog.to_executorch()

    PTE_DIR = Path("pte_files")
    PTE_DIR.mkdir(exist_ok=True)
    pte_path = PTE_DIR / f"resnet18_trial_{trial_id}.pte"

    with open(pte_path, "wb") as f:
        exec_prog.write_to_file(f)

    return str(pte_path)

In [ ]:
import shlex

from EDGE_device.device_specifications import get_adb_path

def move_pte_to_app(pte_path: str, app_name: str):
    """Moves model to the app's private storage on the Android phone."""
    adb = get_adb_path()

    maybe_model_name = re.search(r'([^/\\]+\.pte)$', pte_path)
    model_name = maybe_model_name.group(1) if maybe_model_name else "model.pte"

    subprocess.run([adb, "push", pte_path, "/data/local/tmp/"], check=True, timeout=30,)
    subprocess.run([adb, "shell", "run-as", app_name, "cp", f"/data/local/tmp/{model_name}", "files/model.pte",], check=True, timeout=30,)
    
def move_dataset_to_app(dataset_path: str, app_name: str):
    """Copy dataset into files/dataset/ so MV3Demo sees files/dataset/train|val|test/ (ImageFolder)."""
    adb = get_adb_path()
    dataset_name = os.path.basename(os.path.normpath(dataset_path))
    subprocess.run([adb, "push", dataset_path, "/data/local/tmp/"], check=True, timeout=600)
    src_dot = shlex.quote(f"/data/local/tmp/{dataset_name}/.")
    inner_script = (
        f"DEST=$(ls -d /data/user/*/{app_name}/files 2>/dev/null | head -n1); "
        f'[ -n "$DEST" ] || DEST=/data/data/{app_name}/files; '
        f'test -d "$DEST" && rm -rf "$DEST/dataset" && mkdir -p "$DEST/dataset" && '
        f'cp -R {src_dot} "$DEST/dataset/"'
    )
    remote_line = f"run-as {shlex.quote(app_name)} sh -c {shlex.quote(inner_script)}"
    r = subprocess.run(
        [adb, "shell", remote_line],
        capture_output=True,
        text=True,
        timeout=600,
    )
    if r.returncode != 0:
        raise subprocess.CalledProcessError(r.returncode, [adb, "shell", remote_line], r.stdout, r.stderr)


def clear_edge_metrics_json_on_device(app_name: str) -> None:
    """Remove all metrics_trial_*.json under app files/ (avoids stale reads before a study or trial)."""
    adb = get_adb_path()
    inner_script = (
        f"DEST=$(ls -d /data/user/*/{app_name}/files 2>/dev/null | head -n1); "
        f'[ -n "$DEST" ] || DEST=/data/data/{app_name}/files; '
        f'rm -f "$DEST"/metrics_trial_*.json'
    )
    remote_line = f"run-as {shlex.quote(app_name)} sh -c {shlex.quote(inner_script)}"
    subprocess.run(
        [adb, "shell", remote_line],
        capture_output=True,
        text=True,
        timeout=30,
    )


### Benchmark for the trials on the phone

The functions push the optimised `.pte` and (once) a dataset folder into **Image Classification** (MV3Demo project) private storage (`files/model.pte`, `files/dataset/...`), then start **MainActivity** with action `com.image_classification_app.action.BENCHMARK`. **MaseOptimise** (in the app sources) runs ImageFolder eval and writes **`metrics_trial_<trial_id>.json`** into internal **`files/`** (same private storage as the model). The host polls with `adb exec-out run-as <app_id> cat files/metrics_trial_<id>.json` (requires a **debuggable** build so `run-as` works).

Pipeline: install the app once, set `EDGE_DEVICE_DATASET_PATH` (host folder with `train/`, `val/`, `test/`), set `EDGE_EVAL_SPLIT` if not `val`, then run the study.

Per trial: `adb push` .pte → `run-as` copy to `files/model.pte` → `am start` with extras `split`, `trial_id` → poll/pull metrics JSON.


In [ ]:
from typing import Any

from EDGE_device.device_specifications import get_adb_path

def _metrics_payload_ready(data: dict) -> bool:
    st = data.get("status")
    if st in ("done", "error"):
        return True
    return isinstance(data, dict) and "latency_p95_ms" in data and "top1_acc" in data


def _adb_cat_stdout_not_metrics_json(raw: bytes) -> bool:
    """True when cat failed or file is missing — error text often on stdout with adb returncode 0."""
    if not (raw or b"").strip():
        return True
    if raw.lstrip().startswith(b"cat:"):
        return True
    if b"No such file or directory" in raw:
        return True
    if b"Permission denied" in raw:
        return True
    return False


def edge_benchmark_trial(
    trial_id: int,
    pte_path: str,
    dataset_path: Optional[str] = None,
    split: Optional[str] = None,
    app_name: str = IMAGE_CLASSIFICATION_APP_ID,
    *,
    poll_timeout_sec: Optional[int] = None,
    max_images: Optional[int] = None,
) -> dict[str, Any]:
    """Push → benchmark → metrics. MaseOptimise writes JSON to internal files/; host reads via run-as cat."""

    ds = dataset_path if dataset_path is not None else EDGE_DEVICE_DATASET_PATH
    ev_split = (split or EDGE_EVAL_SPLIT).lower()
    benchmark_action = f"{app_name}.action.BENCHMARK"
    cap = EDGE_BENCHMARK_MAX_IMAGES_FLAG if max_images is None else max_images
    try:
        move_pte_to_app(pte_path=pte_path, app_name=app_name)
        if trial_id == 0 and ds:
            move_dataset_to_app(dataset_path=ds, app_name=app_name)

        adb = get_adb_path()
        am_cmd = [
            adb,
            "shell",
            "am",
            "start",
            "-n",
            f"{app_name}/.MainActivity",
            "-a",
            benchmark_action,
            "-e",
            "split",
            ev_split,
            "-e",
            "trial_id",
            str(trial_id),
        ]
        if cap is not None and cap > 0:
            am_cmd.extend(["--ei", "max_images", str(cap)])
        # Drop stale JSON or the host reads the *previous* benchmark and returns immediately.
        clear_edge_metrics_json_on_device(app_name)
        subprocess.run(am_cmd, check=True, timeout=30)

        return edge_poll_metrics(
            trial_id,
            app_package=app_name,
            timeout=poll_timeout_sec
            if poll_timeout_sec is not None
            else EDGE_METRICS_POLL_TIMEOUT_FLAG,
        )
    except Exception as e:
        print(f"Trial {trial_id} failed: {e}")
        return {
            "top1_acc": 0.0,
            "latency_p95_ms": 999.0,
            "memory_peak_mb": 0.0,
            "num_samples": 0,
            "error": str(e),
            "status": "error",
            "valid_forward_count": 0,
            "decode_failures": 0,
        }


def edge_poll_metrics(
    trial_id: int,
    timeout: Optional[int] = None,
    app_package: str = IMAGE_CLASSIFICATION_APP_ID,
) -> dict[str, Any]:
    """Poll metrics JSON from app internal filesDir via adb (debuggable builds: run-as cat)."""

    timeout = EDGE_METRICS_POLL_TIMEOUT_FLAG if timeout is None else timeout
    remote_rel = f"files/metrics_trial_{trial_id}.json"
    adb = get_adb_path()
    start = time.time()
    last_info_ts = 0.0
    time.sleep(2)
    while time.time() - start < timeout:
        proc: Optional[subprocess.CompletedProcess[bytes]] = None
        raw_out = b""
        try:
            proc = subprocess.run(
                [adb, "exec-out", "run-as", app_package, "cat", remote_rel],
                capture_output=True,
                timeout=30,
            )
            raw_out = proc.stdout or b""
            stripped = raw_out.strip()
            if proc.returncode != 0:
                err_txt = (proc.stderr or b"").decode("utf-8", errors="replace").strip()
                if err_txt:
                    print(f"  [adb] cat rc={proc.returncode}: {err_txt[:400]}")
                time.sleep(2)
                continue
            if not stripped:
                time.sleep(2)
                continue
            if _adb_cat_stdout_not_metrics_json(raw_out):
                now = time.time()
                if now - last_info_ts >= 30:
                    last_info_ts = now
                    elapsed = int(now - start)
                    print(
                        f"  [poll] metrics file not written yet ({elapsed}s / {timeout}s). "
                        "MaseOptimise only creates JSON after the full eval loop finishes "
                        f"(lower EDGE_BENCHMARK_MAX_IMAGES_FLAG for a shorter run). remote={remote_rel!r}"
                    )
                time.sleep(2)
                continue
            try:
                text = stripped.decode("utf-8")
                data = json.loads(text)
            except json.JSONDecodeError as je:
                preview = repr(raw_out[:200])
                print(
                    f"  [poll] JSONDecodeError: {je!s}; rc={proc.returncode}; stdout_preview={preview}"
                )
                time.sleep(2)
                continue

            if not _metrics_payload_ready(data):
                time.sleep(2)
                continue

            err = data.get("error")
            if err and str(err).lower() not in ("null", "none", ""):
                print(f"  [metrics] device error field: {err}")

            st_raw = data.get("status")
            status_str = st_raw if isinstance(st_raw, str) else ""
            out: dict[str, Any] = {
                "top1_acc": float(data.get("top1_acc", 0.0)),
                "latency_p95_ms": float(data.get("latency_p95_ms", 999.0)),
                "memory_peak_mb": float(data.get("memory_peak_mb", 0.0)),
                "num_samples": int(data.get("num_samples", 0)),
                "status": status_str,
                "valid_forward_count": int(data.get("valid_forward_count", 0)),
                "decode_failures": int(data.get("decode_failures", 0)),
            }
            if data.get("num_samples_total_split") is not None:
                out["num_samples_total_split"] = int(data["num_samples_total_split"])
            out["error"] = None if err in (None, "null", "") else str(err)
            out["trial_id_json"] = data.get("trial_id")
            out["split_json"] = data.get("split")
            return out
        except subprocess.TimeoutExpired:
            print("  [poll] adb cat timed out")
            time.sleep(2)
        except Exception as ex:
            print(f"  [poll] {ex!r}")
            time.sleep(2)

    print(
        f"  [poll] timeout after {timeout}s — no readable {remote_rel}. "
        f"Use a debuggable build, or: adb shell run-as {app_package} ls files/"
    )
    return {
        "top1_acc": 0.0,
        "latency_p95_ms": 999.0,
        "memory_peak_mb": 9999.0,
        "num_samples": 0,
        "error": "poll_timeout",
        "status": "error",
        "valid_forward_count": 0,
        "decode_failures": 0,
    }


### 5. Defining the Objective Function

In [ ]:
def objective(trial):
    """Full pipeline: config, model, export, phone, score."""

    config = edge_optuna_config(trial)
    
    model = edge_optimise_model(config, enable_qat=False)
    print("Model is ready for prunes + quantised and ready for pte export")
    
    host_acc = edge_host_val_sanity_check(model)
    if host_acc is not None:
        print(f"  [host sanity] ~top1 over few train batches: {host_acc:.4f}")
        trial.set_user_attr("host_train_sanity_top1", host_acc)
        if host_acc < 0.01:
            print(
                "  [host sanity] WARN: very low — check data path, .pt checkpoint, or prune strength."
            )
    else:
        print("  [host sanity] skipped (Tiny ImageNet train folder missing)")

    pte_path = edge_export_model(model, trial.number, config)

    metrics = edge_benchmark_trial(
        trial.number, pte_path, split=EDGE_EVAL_SPLIT
    )

    acc = float(metrics.get("top1_acc", 0.0))
    latency = float(metrics.get("latency_p95_ms", 999.0))
    memory = float(metrics.get("memory_peak_mb", 9999.0))
    n_samp = int(metrics.get("num_samples", 0))
    v_fwd = int(metrics.get("valid_forward_count", 0))
    dec_fail = int(metrics.get("decode_failures", 0))
    m_err = metrics.get("error")
    m_stat = metrics.get("status", "")

    score = acc - 0.02 * (latency / 30.0) - 0.001 * (memory / 2000.0)

    trial.set_user_attr("top1_acc", acc)
    trial.set_user_attr("latency_ms", latency)
    trial.set_user_attr("memory_mb", memory)
    trial.set_user_attr("num_samples", n_samp)
    trial.set_user_attr("valid_forward_count", v_fwd)
    trial.set_user_attr("decode_failures", dec_fail)
    trial.set_user_attr("metrics_status", m_stat)
    if m_err:
        trial.set_user_attr("metrics_error", m_err)
    for k, v in config.items():
        trial.set_user_attr(k, v)

    print(
        f"  → acc={acc:.3f}, lat={latency:.1f}ms, score={score:.3f} | "
        f"samples={n_samp}, forwards={v_fwd}, decode_fail={dec_fail}, status={m_stat!r}"
    )
    if m_err:
        print(f"  → metrics error field: {m_err}")
    return score

### 6. Optuna Study

In [ ]:
# Create + run study
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="resnet18_edge_opt",
    storage="sqlite:///optuna_resnet18.db",  # Save progress
    load_if_exists=True,
)

# Run 50 trials
study.optimize(objective, n_trials=50)

print(f"\nBest trial #{study.best_trial.number}")
print(f"   Score: {study.best_value:.3f}")
print("   Config:", study.best_params)

# Quick viz

optuna.visualization.plot_optimization_history(study).show()


### Extract the best model

In [ ]:
# Get best config
best_trial = study.best_trial
best_config = {
    k: v
    for k, v in best_trial.user_attrs.items()
    if k
    in (
        "prune_ratio",
        "quant_bits",
        "quant_train_mode",
        "backend",
        "fusion",
        "quant_config_global",
        "use_mixed_delegation",
        "delegation_plan",
    )
}

# Rebuild + export best model
best_model = edge_optimise_model(best_config)
final_pte = edge_export_model(best_model, 999, best_config)

print(f"Best model: {final_pte}")
print("Deploy with: adb push", final_pte, "/sdcard/resnet18_best.pte")
